# Import data
データ取り込みを行う。 PostgresSQL PgVector 想定。

In [ ]:
import os
import dotenv

dotenv.load_dotenv()
ENV_PG_CONNECTION_STRING = os.getenv("ENV_PG_CONNECTION_STRING")
ENV_GEMINI_API_KEY = os.getenv("ENV_GEMINI_API_KEY")

# Obsidian Vault

## Extract

In [ ]:
from pathlib import Path
from agent_assistant.loader.obsidian import VaultLoader

# Vault から読み込み
vault_path = Path("../docs/dataset_obsidian/")
loader = VaultLoader(vault_path)
docs = loader.load()

print(f"{len(docs)} 件のノートを読み込みました")

## Load (DuckDB)

In [ ]:
from pathlib import Path

# 保存先ディレクトリを作成
vault_db_path = Path("../tests/data/vault_db").resolve()
vault_db_path.mkdir(exist_ok=True)

In [ ]:
from sqlalchemy import create_engine, text
from agent_assistant.entities.duckdb import ObsidianVaultBase, ObsidianVaultRawEntity
from agent_assistant.loader.obsidian import VaultDb

# DB へ取り込み
path = vault_db_path / "entity.duckdb"
sa_engine = create_engine(f"duckdb:///{path}")

with sa_engine.connect() as sess:
    sess.execute(text("create schema if not exists assets;"))
    sess.commit()

ObsidianVaultBase.metadata.create_all(sa_engine)
VaultDb.sync(docs, sa_engine, ObsidianVaultRawEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
from pprint import pprint
from sqlalchemy import text

with sa_engine.connect() as sess:
    res = sess.execute(text("select * from entity.assets.obsidian_vault_raw limit 3"))
    pprint(res.all())

In [ ]:
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from agent_assistant.entities.duckdb import ObsidianVaultRawEntity
from agent_assistant.retriever.obsidian_llama import ObsidianLlamaRetriever
from agent_assistant.utils.store_factory import DuckDBStoreContext

# レトリーバーを作成
store_ctx = DuckDBStoreContext(vault_db_path)
sa_engine = store_ctx.get_engine()
obsidian_retriever = ObsidianLlamaRetriever(
    sa_engine,
    store_ctx,
    "obsidian_vault_docstore",
    "obsidian_vault_vectors",
    GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=ObsidianVaultRawEntity,
)

In [ ]:
# obsidian_retriever.sync_chunks()

In [ ]:
# list(store_ctx._conn.execute("SHOW DATABASES").fetchall())
list(store_ctx._conn.execute("SHOW TABLES").fetchall())
# list(store_ctx._conn.execute("SELECT * FROM obsidian_vault_docstore LIMIT 10").fetchall())
# list(store_ctx._conn.execute("SELECT * FROM obsidian_vault_vectors LIMIT 10").fetchall())

## Load (PostgresSQL)

In [ ]:
from langchain_postgres import PGEngine
from sqlalchemy import create_engine

# エンジン初期化
assert ENV_PG_CONNECTION_STRING is not None
assert ENV_GEMINI_API_KEY is not None

sa_engine = create_engine(ENV_PG_CONNECTION_STRING)
pg_engine = PGEngine.from_connection_string(ENV_PG_CONNECTION_STRING, pool_size=5)

In [ ]:
from agent_assistant.loader.obsidian import VaultDb
from agent_assistant.entities.postgres import ObsidianVaultBase, ObsidianVaultRawEntity

# DB へ取り込み
ObsidianVaultBase.metadata.create_all(sa_engine)
VaultDb.sync(docs, sa_engine, ObsidianVaultRawEntity)

print(f"{len(docs)} 件のノートを読み込みました")

In [ ]:
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding
from agent_assistant.utils.store_factory import PostgresStoreContext
from agent_assistant.entities.postgres import ObsidianVaultRawEntity
from agent_assistant.retriever.obsidian_llama import ObsidianLlamaRetriever

assert ENV_PG_CONNECTION_STRING is not None
assert ENV_GEMINI_API_KEY is not None

store_ctx = PostgresStoreContext(ENV_PG_CONNECTION_STRING, schema_name="app")
sa_engine = store_ctx.get_engine()
obsidian_retriever = ObsidianLlamaRetriever(
    sa_engine,
    store_ctx,
    "obsidian_vault_docstore",
    "obsidian_vault_vectors",
    GoogleGenAIEmbedding(
        model_name="gemini-embedding-001",
        api_key=ENV_GEMINI_API_KEY,
    ),
    embed_dim=3072,
    vault_entity=ObsidianVaultRawEntity,
)

In [ ]:
# obsidian_retriever.sync_chunks()

## 動作確認

In [ ]:
from llama_index.core.vector_stores.types import (
    VectorStoreQuery, VectorStoreQueryMode,
    MetadataFilters, MetadataFilter, FilterOperator
)
from llama_index.embeddings.google_genai import GoogleGenAIEmbedding


ember = GoogleGenAIEmbedding(api_key=os.getenv("ENV_GEMINI_API_KEY"))
query_res = obsidian_retriever._vector_store.query(
    VectorStoreQuery(
        query_embedding=ember.get_query_embedding("プロンプトエンジニアリング"),
        similarity_top_k=5,
        mode=VectorStoreQueryMode.MMR,
        mmr_threshold=0.5,
        filters=MetadataFilters(
            filters=[
                MetadataFilter(key="path", value="03_Structure/", operator=FilterOperator.TEXT_MATCH)
            ]
        )
    )
)
assert query_res.nodes is not None
for item in query_res.nodes:
    print(item.text)  # pyright: ignore[reportAttributeAccessIssue]
    print("==================")